In [3]:
import pandas as pd
import ast

In [ ]:
df = pd.read_csv("../../data/raw/movies_metadata.csv", low_memory=False)
df.shape

In [ ]:
df[[
    "id",
    "original_title",
    "release_date",
    "genres",
    "production_countries",
    "spoken_languages"
]].head()

In [ ]:
df[[
    "id",
    "original_title",
    "genres",
    "production_countries",
    "spoken_languages"
    ]].isna().sum()

In [ ]:
nested_columns = [
    "genres",
    "production_countries",
    "spoken_languages"
]

for column in nested_columns:
    print(column, (df[column] == "[]").sum())

In [8]:
def check_nested_value(value):
    if pd.isna(value):
        return "missing"
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"
    
    if parsed == []:
        return "empty_list"
    
    if not isinstance(parsed, list):
        return "not_a_list"
    
    return "valid"

In [ ]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_nested_value).value_counts())
    print()

In [ ]:
country_status = df["production_countries"].apply(check_nested_value)

df.loc[
    country_status == "not_a_list",
    ["id", "original_title", "production_countries"]
]

In [ ]:
pd.set_option("display.max_columns", None)

df.loc[[19730, 29503, 35587]]

In [12]:
def check_inner_structure(value):
    if pd.isna(value):
        return "missing"

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return "malformed"

    if parsed == []:
        return "empty_list"

    if not isinstance(parsed, list):
        return "not_a_list"

    for item in parsed:
        if not isinstance(item, dict):
            return "item_not_dictionary"

        if "name" not in item:
            return "missing_name"

        if not isinstance(item["name"], str) or item["name"].strip() == "":
            return "empty_name"

    return "valid"

In [ ]:
for column in nested_columns:
    print(column)
    print(df[column].apply(check_inner_structure).value_counts())
    print()

In [ ]:
language_status = df["spoken_languages"].apply(check_inner_structure)

df.loc[
    language_status == "empty_name",
    ["id", "original_title", "spoken_languages"]
].head(10)

In [15]:
def extract_names(value):
    if pd.isna(value):
        return pd.NA

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return pd.NA

    if not isinstance(parsed, list) or not parsed:
        return pd.NA

    names = []

    for item in parsed:
        if isinstance(item, dict):
            name = item.get("name")

            if isinstance(name, str):
                name = name.strip()

                if name and name not in names:
                    names.append(name)

    if not names:
        return pd.NA

    return "|".join(names)

In [ ]:
cleaned_df = df[[
    "id",
    "original_title",
    "genres",
    "production_countries",
    "spoken_languages"
]].copy()

for column in nested_columns:
    cleaned_df[column] = cleaned_df[column].apply(extract_names)

cleaned_df.head()

In [ ]:
numeric_ids = pd.to_numeric(df["id"], errors="coerce")

invalid_ids = df[numeric_ids.isna()][
    ["id", "original_title"]
]

invalid_ids

In [ ]:
df[df["id"].duplicated(keep=False)][
    ["id", "original_title"]
].sort_values("id")

In [ ]:
print("Duplicate rows:", df["id"].duplicated(keep=False).sum())
print("Unique duplicated IDs:", df.loc[df["id"].duplicated(keep=False), "id"].nunique())

In [ ]:
duplicate_rows = cleaned_df[
    cleaned_df["id"].duplicated(keep=False)
].sort_values("id")

duplicate_rows

In [ ]:
duplicate_check = (
    cleaned_df
    .groupby("id", dropna=False)
    .nunique(dropna=False)
)

duplicate_check[
    (duplicate_check > 1).any(axis=1)
]

In [ ]:
cleaned_df = cleaned_df.drop_duplicates(subset="id", keep="first")

cleaned_df.shape

In [ ]:
cleaned_df = cleaned_df[
    pd.to_numeric(cleaned_df["id"], errors="coerce").notna()
].copy()

cleaned_df.shape

In [ ]:
blank_titles = cleaned_df[
    cleaned_df["original_title"].astype(str).str.strip() == ""
]

blank_titles

In [ ]:
cleaned_df[[
    "genres",
    "production_countries",
    "spoken_languages"
]].isna().sum()

In [ ]:
def has_duplicate_names(value):
    if pd.isna(value):
        return False

    names = value.split("|")
    return len(names) != len(set(names))

for column in nested_columns:
    print(column, cleaned_df[column].apply(has_duplicate_names).sum())

In [ ]:
cleaned_df["id"].duplicated().sum()

In [ ]:
for column in nested_columns:
    print(column, (cleaned_df[column] == "").sum())

## Preprocessing Decisions

For my part of the preprocessing, I worked with `id`, `original_title`, `genres`, `production_countries`, and `spoken_languages`.

For the three nested columns, I used `ast.literal_eval` to safely turn the values into lists and dictionaries. I then extracted the `name` from each dictionary so that the final data would be easier to read and use. When a movie had multiple values, I separated them using `|`.

I also removed repeated values within the same movie and stripped extra whitespace from the names. Empty lists, missing values, or values that did not have any usable names were kept as missing values. Some of the language entries had a language code but a blank `name`, so I skipped those blank names instead of replacing them with the code since the goal was to keep readable language names.

While checking the `id` column, I found 3 rows with invalid IDs that were actually dates. Looking at the full rows showed that the data in those rows was shifted/malformed, so I removed those records.

I also found 29 movie IDs that appeared more than once, involving 59 rows total. I checked the cleaned values for these duplicate IDs and they were the same, so I kept the first occurrence of each ID to make sure there was only one row per movie.

Finally, I checked that there were no duplicate IDs, repeated values within the nested columns, or empty strings left in the cleaned data.